# Autoencoder Anomaly Detection for TLS Profiling

This notebook evaluates the autoencoder implementation from `tls_profiling.autoencoder` under the same malware-dataset protocol used by the classical baselines in `notebooks/baselines`. For each target label, the model is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic using reconstruction error as the anomaly score.


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The notebook evaluates each group individually and all group combinations to mirror the baseline ablation study.


## Evaluation

For each label in `system`, `unknown`, and `malware`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses the same three disjoint splits as the baseline notebooks:

- `train`: only samples from the selected normal class, used to fit the autoencoder
- `validation`: mixed traffic, used to choose the anomaly-score threshold that maximizes `F1`
- `test`: held-out mixed traffic, used only for final reporting

Because the neural model needs a reconstruction-monitor split during fitting, the notebook carves a small internal holdout only from the selected normal training traffic and keeps the public validation split for threshold tuning, so the outer train/validation/test protocol remains aligned with the baselines.

Reported metrics:

- `AUC-ROC`: how well reconstruction error ranks the selected class against the rest over all thresholds
- `AP`: average precision of the anomaly ranking
- `F1`: test-set score obtained with the threshold selected on the validation split


In [ ]:
import sys
sys.path.append('../../src')

  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.1.2-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (7.4 kB)
  Using cached h5py-3.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cache

In [6]:
FLOW = ['bs', 'ps', 'br', 'pr', 'td']
CTLS = ['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv']
STLS = ['tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv']
REC = ['tls.rec']


In [7]:
from pathlib import Path

import pandas as pd
from tls_profiling.preprocessing import build_and_fit_preprocessor

data_dir = Path('../data-ext')
# data_dir = Path('../data')

print(f'Loading extracted features from {data_dir}.')
df_train = pd.read_parquet(data_dir / 'malware_train.parquet')
df_val = pd.read_parquet(data_dir / 'malware_val.parquet')
df_test = pd.read_parquet(data_dir / 'malware_test.parquet')
df_train_label = pd.read_parquet(data_dir / 'malware_train_labels.parquet')
df_val_label = pd.read_parquet(data_dir / 'malware_val_labels.parquet')
df_test_label = pd.read_parquet(data_dir / 'malware_test_labels.parquet')

preprocessor = build_and_fit_preprocessor(df_train)
print('Built preprocessor from df_train.')


Loading extracted features from ../data-ext.
Built preprocessor from df_train.


In [ ]:
from itertools import combinations
import random

import numpy as np
import tensorflow as tf
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split

from tls_profiling.autoencoder import (
    build_conv_dense_autoencoder,
    train_autoencoder_model,
    compute_reconstruction_error,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

AE_CONFIG = {
    'encoding_dim': 16,
    'conv_input_size': 20,
    'intermediate_dim': 64,
    'max_epochs': 50,
    'batch_size': 128,
    'lr': 1e-3,
    'loss': 'mse',
    'early_stopping_patience': 8,
    'train_holdout_ratio': 0.2,
}

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def choose_f1_threshold(y_true, anomaly_score):
    precision, recall, thresholds = precision_recall_curve(y_true, anomaly_score)

    if len(thresholds) == 0:
        return float('inf')

    f1_scores = (2 * precision[:-1] * recall[:-1]) / np.clip(
        precision[:-1] + recall[:-1],
        a_min=1e-12,
        a_max=None,
    )
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])

def to_numpy_float32(x):
    return x.to_numpy(dtype=np.float32, copy=False)

def split_train_for_autoencoder(x_train):
    if len(x_train) < 3:
        return x_train, x_train

    x_fit, x_monitor = train_test_split(
        x_train,
        test_size=AE_CONFIG['train_holdout_ratio'],
        random_state=SEED,
        shuffle=True,
    )
    return x_fit, x_monitor

def choose_encoding_dim(input_dim):
    return max(2, min(AE_CONFIG['encoding_dim'], max(2, input_dim // 2)))

def choose_conv_input_size(input_dim):
    if input_dim <= 2:
        return 1
    return max(1, min(AE_CONFIG['conv_input_size'], input_dim - 1))

evaluation_cache = {}

def evaluate(feature_set, normal_label):
    cache_key = (tuple(feature_set), normal_label)
    if cache_key in evaluation_cache:
        return evaluation_cache[cache_key]

    x_train = df_train.loc[df_train_label['category'] == normal_label].reset_index(drop=True)
    x_val = df_val
    y_val = (df_val_label['category'] != normal_label).astype(int).to_numpy()
    x_test = df_test
    y_test = (df_test_label['category'] != normal_label).astype(int).to_numpy()

    x_train_transformed = select_feature_set(preprocessor.transform(x_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(x_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(x_test), feature_set)

    x_train_np = to_numpy_float32(x_train_transformed)
    x_val_np = to_numpy_float32(x_val_transformed)
    x_test_np = to_numpy_float32(x_test_transformed)

    x_fit, x_monitor = split_train_for_autoencoder(x_train_np)

    input_dim = x_fit.shape[1]
    encoding_dim = choose_encoding_dim(input_dim)
    conv_input_size = choose_conv_input_size(input_dim)

    tf.keras.backend.clear_session()

    models = build_conv_dense_autoencoder(
        input_dim=input_dim,
        encoding_dim=encoding_dim,
        conv_input_size=conv_input_size,
        intermediate_dim=AE_CONFIG['intermediate_dim'],
    )

    history = train_autoencoder_model(
        models=models,
        x_train=x_fit,
        x_val=x_monitor,
        max_epochs=AE_CONFIG['max_epochs'],
        batch_size=AE_CONFIG['batch_size'],
        lr=AE_CONFIG['lr'],
        loss=AE_CONFIG['loss'],
        early_stopping_patience=AE_CONFIG['early_stopping_patience'],
        verbose=0,
    )

    val_scores = compute_reconstruction_error(models.autoencoder, x_val_np, metric='mse')
    threshold = choose_f1_threshold(y_val, val_scores)
    anomaly_score = compute_reconstruction_error(models.autoencoder, x_test_np, metric='mse')

    result = {
        'y_test': y_test,
        'anomaly_score': anomaly_score,
        'threshold': threshold,
        'history': history,
        'input_dim': input_dim,
        'encoding_dim': encoding_dim,
        'conv_input_size': conv_input_size,
    }
    evaluation_cache[cache_key] = result
    return result

def debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score):
    feature_set_name = '_'.join(feature_set)
    class_label_name = normal_label

    output_dir = Path('tmp')
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / f'ae_{class_label_name}_{feature_set_name}.csv'
    pd.DataFrame({
        'y_test': y_test,
        'y_pred': y_pred,
        'anomaly_score': anomaly_score,
    }).to_csv(output_path, index=False)

def compute_scores(feature_set, normal_label):
    result = evaluate(feature_set, normal_label)
    y_test = result['y_test']
    anomaly_score = result['anomaly_score']
    threshold = result['threshold']

    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)

    debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score)
    return {
        'auc_score': auc,
        'ap_score': ap,
        'f1_score': f1,
        'threshold': threshold,
        'input_dim': result['input_dim'],
        'encoding_dim': result['encoding_dim'],
        'conv_input_size': result['conv_input_size'],
    }

def plot_curves(feature_set, normal_label):
    result = evaluate(feature_set, normal_label)
    y_test = result['y_test']
    anomaly_score = result['anomaly_score']

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name='AE PR-AUC',
        ax=axes[0],
    )
    axes[0].set_title(f'{normal_label} Precision-Recall')

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name='AE AUC-ROC',
        ax=axes[1],
    )
    axes[1].set_title(f'{normal_label} ROC Curve')

    plt.tight_layout()
    plt.show()


I0000 00:00:1776406552.309532  307894 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776406553.890658  307894 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776406573.528816  307894 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
feature_groups = {
    'FLOW': FLOW,
    'CTLS': CTLS,
    'STLS': STLS,
    'REC': REC,
}

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ', '.join(group_combo)

        for label in ['system', 'unknown', 'malware']:
            scores = compute_scores(selected_features, label)

            rows.append({
                'FeatureSet': feature_set_name,
                'ClassLabel': label,
                'AucScore': scores['auc_score'],
                'ApScore': scores['ap_score'],
                'F1Score': scores['f1_score'],
                'Threshold': scores['threshold'],
                'InputDim': scores['input_dim'],
                'EncodingDim': scores['encoding_dim'],
                'ConvInputSize': scores['conv_input_size'],
            })

results_df = pd.DataFrame(
    rows,
    columns=[
        'FeatureSet',
        'ClassLabel',
        'AucScore',
        'ApScore',
        'F1Score',
        'Threshold',
        'InputDim',
        'EncodingDim',
        'ConvInputSize',
    ],
)
display(results_df)


E0000 00:00:1776406614.584990  307894 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1776407768.829664  307894 cpu_allocator_impl.cc:82] Allocation of 217479600 exceeds 10% of free system memory.
W0000 00:00:1776407769.305797  307894 cpu_allocator_impl.cc:82] Allocation of 217479600 exceeds 10% of free system memory.
W0000 00:00:1776411716.738667  307894 cpu_allocator_impl.cc:82] Allocation of 228353580 exceeds 10% of free system memory.
W0000 00:00:1776411717.174403  307894 cpu_allocator_impl.cc:82] Allocation of 228353580 exceeds 10% of free system memory.
W0000 00:00:1776415630.039104  307894 cpu_allocator_impl.cc:82] Allocation of 332743788 exceeds 10% of free system memory.


## System Profile

The table below isolates the autoencoder results for the `system` profile across all feature-group combinations. Lower reconstruction error on `system` traffic and higher error on the remaining categories should translate into stronger ranking metrics here.


In [ ]:
system_df = results_df[results_df['ClassLabel'] == 'system'].sort_values('F1Score', ascending=False)
display(system_df)


## Malware Profile

This section shows the same evaluation when `malware` is treated as the in-profile class. In this setup, the autoencoder answers whether malware traffic itself forms a stable enough profile to reconstruct better than `system` and `unknown` traffic.


In [ ]:
malware_df = results_df[results_df['ClassLabel'] == 'malware'].sort_values('F1Score', ascending=False)
display(malware_df)


## Unknown Category

The `unknown` label is still a residual group rather than a clean semantic class, so these results are best interpreted as a stress test of how broadly the autoencoder can profile heterogeneous traffic under the same baseline protocol.


In [ ]:
unknown_df = results_df[results_df['ClassLabel'] == 'unknown'].sort_values('F1Score', ascending=False)
display(unknown_df)
